In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.datasets as datasets
from torch.utils.data import DataLoader

In [ ]:
data_dir = './data'

transform = T.Compose([
    T.Resize(224),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
])

test_dataset  = datasets.CIFAR10(root=data_dir, train=False, download=True, transform=transform)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

In [ ]:
import torchvision.models as M

# Move to CPU/GPU device -- DO NOT MODIFY
device = "cuda" if torch.cuda.is_available() else "cpu"

## YOUR SOLUTION HERE ##
# Load ResNet18 and adapt for CIFAR-10 
model_resnet18 = M.resnet18(weights=None)
in_feats = model_resnet18.fc.in_features
model_resnet18.fc = nn.Linear(in_feats, 10)

# Load pre-trained weights fine-tuned on CIFAR-10
state_dict_resnet18 = torch.load("models/resnet18_cifar10.pt", map_location=device, weights_only=True)
model_resnet18.load_state_dict(state_dict_resnet18)

# Move to device and set to evaluation mode
model_resnet18 = model_resnet18.to(device)
model_resnet18.eval()

# Show output
from custom_torchinfo import custom_summary
custom_summary(model_resnet18, input_size=(1, 3, 224, 224))

In [ ]:
## YOUR SOLUTION HERE ##
def predict(model, dataloader, device="cpu"):
    model.eval()
    all_logits = []
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch_X, batch_y in dataloader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            # Forward pass
            logits = model(batch_X)

            # Convert logits to predicted labels
            probs = torch.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)

            # Save predictions
            all_logits.append(logits.cpu().numpy())
            all_predictions.append(preds.cpu().numpy())
            all_labels.append(batch_y.cpu().numpy())

    # Join predictions
    y_logits = np.concatenate(all_logits, axis=0)
    y_pred   = np.concatenate(all_predictions, axis=0)
    y_true   = np.concatenate(all_labels, axis=0)

    return y_logits, y_pred, y_true

# Generate test predictions
y_logits_resnet18, y_pred_resnet18, y_true = predict(model_resnet18, test_loader, device=device)

# Show output
print("First 10 predictions:", y_pred_resnet18[:10])
print("First 10 labels:", y_true[:10])

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

## YOUR SOLUTION HERE ##
test_accuracy_resnet18 = accuracy_score(y_true, y_pred_resnet18)
report_resnet18 = classification_report(y_true, y_pred_resnet18, target_names=test_dataset.classes)

# Show output
print(f"Accuracy: {test_accuracy_resnet18:.4f}")
print(report_resnet18)

In [ ]:
import torchvision.models as M

model_mobilenet = torch.load("./models/model_mobilenet.pth", map_location=device, weights_only=True)
in_feats = model_mobilenet.classifier[-1].in_features
model_mobilenet.classifier[-1] = torch.nn.Linear(in_feats, 10)  # Assuming 10 classes

state_dict = torch.load("./models/model_mobilenet.pth")
model_mobilenet.load_state_dict(state_dict)
model_mobilenet  = model_mobilenet.to("cuda" if torch.cuda.is_available() else "cpu")

from custom_torchinfo import custom_summary
custom_summary(model_mobilenet, input_size=(1, 3, 224, 224))

y_logits_mobilenet, y_pred_mobilenet, y_true = predict(model_mobilenet, test_loader)
test_accuracy_mobilenet = (y_pred_mobilenet == y_true).float().mean().item()
report_mobilenet = classification_report(y_true, y_pred_mobilenet)

In [ ]:
import time

# Grab the first image in the testing dataset
batch_X, batch_y = next(iter(test_loader))
x1 = batch_X[0].unsqueeze(0).to(device)

def measure_latency(model, x, iters=30):
    """Return average latency (ms) per forward pass."""
    model.eval()
    start = time.perf_counter()
    with torch.inference_mode():
        for _ in range(iters):
            _ = model(x)
            if x.device.type == "cuda":
                torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    return (elapsed / iters) * 1e3  # ms

def measure_gpu_memory(model, x):
    """Return current and peak GPU memory in MB after one forward."""
    torch.cuda.empty_cache()                 
    torch.cuda.reset_peak_memory_stats()
    
    model.eval()
    with torch.inference_mode():
        y = model(x) # Optional - delete predictions afterwards
        
    torch.cuda.synchronize()
    current = torch.cuda.memory_allocated() / (1024**2)
    peak    = torch.cuda.max_memory_allocated() / (1024**2)
    return current, peak

# Measure latency
latency_mobilenet = measure_latency(model_mobilenet, x1, iters=30)
latency_resnet18 = measure_latency(model_resnet18, x1, iters=30)

# Measure GPU allocation
memory_mobilenet = measure_gpu_memory(model_mobilenet, x1)
memory_resnet18 = measure_gpu_memory(model_resnet18, x1)

# Show output
print("Latency Comparison")
warmup  = measure_latency(model_mobilenet, x1, iters=30)
print("MobileNetV3-Small:", f"{latency_mobilenet:.4f} ms")
print("ResNet-18:",         f"{latency_resnet18:.4f} ms")
print("====================================================================")

print("Memory Usage Comparison")
print(f"MobileNetV3-Small: [GPU Memory] Current allocated: {memory_mobilenet[0]:.2f} MB | Peak during forward: {memory_mobilenet[1]:.2f} MB")
print(f"ResNet-18: [GPU Memory] Current allocated: {memory_resnet18[0]:.2f} MB | Peak during forward: {memory_resnet18[1]:.2f} MB")
print("====================================================================")

print("Classification Performance Comparison")
print("MobileNetV3-Small Performance:")
print(f"Accuracy: {test_accuracy_mobilenet:.4f}")
print(report_mobilenet)
print("ResNet-18 Performance:")
print(f"Accuracy: {test_accuracy_resnet18:.4f}")
print(report_resnet18)

In [ ]:
import gc, torch

model_mobilenet.to("cpu")
model_resnet18.to("cpu")

del model_mobilenet, model_resnet18
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()